In [ ]:
using Revise
using Pkg

ENV["PYTHON"] = Sys.which("python")
ENV["PYCALL_JL_RUNTIME_PYTHON"] = Sys.which("python")
Pkg.build("PyCall")
using FileIO
using JLD2
using PyCall
pyimport("sys")."path" |> x -> pushfirst!(x, "../Python-RVO2/build/lib.linux-x86_64-3.6")

push!(LOAD_PATH,"../src")
include("../src/DistributionallyRobust.jl")
using .DistributionallyRobust

In [ ]:
include("$(@__DIR__)/../scripts/default_params/params_drc_data_trajectron.jl");

epsilon = 0.05;
test_data_name = "eth_test.pkl";     
# test_data_name = "hotel_test.pkl";                                                  # test data set name
# start_time_idx = 401;                                                               # start time index in test data
test_scene_id = 0;                                                                    # test data id

ego_pos_init_vec = [0.1, -8.7] 
ego_pos_goal_vec= [0.1, 2.7] 

target_speed = 2.0;                                                                   # target speed [m/s]     
sim_horizon = 10.0
deterministic = true  
num_samples = 1  
prediction_steps = 2
include("$(@__DIR__)/../scripts/parameter_setup_drc.jl");

In [ ]:
scene_loader, controller, w_init, measurement_schedule, target_trajectory, target_speed =
    controller_setup(scene_param,
                    predictor_param,
                    prediction_device=prediction_device,
                    cost_param=cost_param,
                    cnt_param=cnt_param,
                    dtc=dtc,
                    run_id=35,
                    ego_pos_init_vec=ego_pos_init_vec,
                    ego_pos_goal_vec=ego_pos_goal_vec,
                    target_speed=target_speed,
                    sim_horizon=sim_horizon,                    
                    verbose=true);

In [ ]:
result, ~, ~, comp_time_list = evaluate(scene_loader, controller, w_init, ego_pos_goal_vec,
                  target_speed, measurement_schedule, target_trajectory,
                  pos_error_replan,safety_distance,max_MPC_iters,35);

In [ ]:
result.total_cnt_cost

In [ ]:
result.total_pos_cost

In [ ]:
result.total_col_cost

In [ ]:
result.total_col

In [ ]:
result.total_cnt_cost + result.total_pos_cost + result.total_col_cost

In [ ]:
minimum([minimum(vcat([norm(get_position(w.e_state) - ap) for ap in values(w.ap_dict)], Inf))
                          for w in result.w_history])

In [ ]:
using CSV
using JLD2
dir_path = "./ETH"
if !isdir(dir_path)
    mkdir(dir_path)
end

for (idx, predict_dict) in enumerate(result.prediction_dict_history)
    if predict_dict != nothing
        CSV.write("./ETH/predict_dict_$idx.csv", predict_dict)
    end
end

prediction_dict_history = result.prediction_dict_history
@save "./ETH/prediction_dict_history.jld2" prediction_dict_history

position_history = [get_position(w.e_state) for w in result.w_history]
velocity_history = [get_velocity(w.e_state) for w in result.w_history]
open("./ETH/position_history.csv", "w") do file
    for row in position_history
        # Convert the row to a comma-separated string and write to file
        println(file, join(row, ","))
    end
end
open("./ETH/velocity_history.csv", "w") do file
    for row in velocity_history
        # Convert the row to a comma-separated string and write to file
        println(file, join(row, ","))
    end
end

In [ ]:
println(controller.Goal_reached)